# Test Augmented Tables
Validate the BattingAugmented and PitchingAugmented CSVs from `data/`.

In [ ]:
import pandas as pd

batting = pd.read_csv('data/BattingAugmented.csv')
pitching = pd.read_csv('data/PitchingAugmented.csv')
people = pd.read_csv('data/People.csv')

print(f'Batting: {len(batting)} rows, {batting["WAR"].notna().sum()} with WAR')
print(f'Pitching: {len(pitching)} rows, {pitching["WAR"].notna().sum()} with WAR')

## Spot-check: Aaron Judge (multi-year)

In [ ]:
judge = batting[batting['playerID'] == 'judgeaa01'][['playerID', 'yearID', 'teamID', 'G', 'AB', 'HR', 'RBI', 'WAR']]
judge.sort_values('yearID')

## Spot-check: Shohei Ohtani (batting + pitching)

In [ ]:
ohtani_bat = batting[batting['playerID'] == 'ohtansh01'][['playerID', 'yearID', 'teamID', 'G', 'AB', 'HR', 'WAR']]
print('Ohtani Batting:')
display(ohtani_bat.sort_values('yearID'))

ohtani_pit = pitching[pitching['playerID'] == 'ohtansh01'][['playerID', 'yearID', 'teamID', 'G', 'W', 'L', 'ERA', 'WAR']]
print('Ohtani Pitching:')
display(ohtani_pit.sort_values('yearID'))

## Multi-team player check
Players who switched teams mid-season should have separate rows with WAR for each team.

In [ ]:
# Find players with multiple stints in a single year
multi = batting.groupby(['playerID', 'yearID']).filter(lambda g: len(g) > 1)
multi_with_war = multi[multi['WAR'].notna()]
print(f'Multi-team rows: {len(multi)}, with WAR: {len(multi_with_war)}')

# Show a specific example
example_player = multi_with_war.groupby('playerID').first().index[0]
example = multi[multi['playerID'] == example_player][['playerID', 'yearID', 'teamID', 'stint', 'G', 'AB', 'HR', 'WAR']]
print(f'\nExample: {example_player}')
display(example.sort_values(['yearID', 'stint']))

## Rows missing WAR
Check what's missing — likely low-AB players not in BBRef WAR tables.

In [ ]:
missing = batting[batting['WAR'].isna()]
print(f'Missing WAR: {len(missing)} rows')
print(f'Mean AB for missing: {missing["AB"].mean():.1f}')
print(f'Mean AB for present: {batting[batting["WAR"].notna()]["AB"].mean():.1f}')
print(f'\nAB distribution of missing WAR rows:')
missing['AB'].describe()

## WAR leaders by year

In [ ]:
bat_leaders = batting.dropna(subset=['WAR']).merge(
    people[['playerID', 'nameFirst', 'nameLast']], on='playerID'
)
bat_leaders['Name'] = bat_leaders['nameFirst'] + ' ' + bat_leaders['nameLast']

# Top 5 batting WAR per year
for year in sorted(batting['yearID'].unique()):
    top = bat_leaders[bat_leaders['yearID'] == year].nlargest(5, 'WAR')[['Name', 'teamID', 'WAR']]
    print(f'\n--- {year} Batting WAR Leaders ---')
    print(top.to_string(index=False))